In [1]:
import os
from pathlib import Path
import logging
from fastwarc import ArchiveIterator, WarcRecordType
import gzip 
from itertools import islice
from cs336_data.raw_data_converter import RawDataConverter
from cs336_data.lang_id.language_identifier import LanguageIdentifier

WARC_EXAMPLES_PATH = '/home/eugencutic/cs336-assignment4-data/local-shared-data/CC/example.warc.gz'
WET_EXAMPLES_PATH = '/home/eugencutic/cs336-assignment4-data/local-shared-data/CC/example.warc.wet.gz'
EXAMPLES_OUTPUT_DIR = Path('./local-shared-data/CC/temp/')

os.makedirs(EXAMPLES_OUTPUT_DIR, exist_ok=True)

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

lang_id = LanguageIdentifier()

In [2]:
start = 0
stop = 5000

texts = []
failed_in_extraction = 0

with gzip.open(WARC_EXAMPLES_PATH) as warc_zip:
    warc_iter = islice(
        iter(
            rec for rec in ArchiveIterator(warc_zip)
            if rec.record_type == WarcRecordType(4) and rec.http_content_type == 'text/html'
        ),
        start,
        stop
    )

    for rec in warc_iter:
        uri = str(rec.headers.get('WARC-Target-URI', ''))
        content = rec.reader.read()

        try:
            converted = RawDataConverter.extract_text(content)
            texts.append((uri,converted))
        except Exception:
            logger.warning(f'Doc with URI {uri} failed in text extraction.')
            failed_in_extraction += 1

logger.info(f"{failed_in_extraction} docs failed in text extraction")

INFO:__main__:86 docs failed in text extraction


In [3]:
from random import Random

seed = 42
rng = Random(x=42)

sample_count = 50
sampled_texts = []

for i in rng.sample(range(len(texts)), k=sample_count):
    sampled_texts.append(texts[i])

print(sampled_texts[0][1])

Barry Dunleavy logo

Default Menu

  • Home
  • Map Search
  • Sellers
  • Services
    • SWFL Market Statistics
    • What is my Property worth?
    • Mortgage Calculator
    • Vacation Rentals
    • Hurricane Preparedness
  • Top Neighborhoods
    • Collier County
    • Golf Communities
  • About
  • Contact
Close Button
Log InSign UpMy Account
  • Home
  • Map Search
  • Sellers
  • Services
    • SWFL Market Statistics
    • What is my Property worth?
    • Mortgage Calculator
    • Vacation Rentals
    • Hurricane Preparedness
  • Top Neighborhoods
    • Collier County
    • Golf Communities
  • About
  • Contact
  • Saved Searches
  • Favorites
Account
Sign Up
or
Log In
239-877-6445

Hamburger Menu

  • Home
  • Map Search
  • Sellers
  • Services
    • SWFL Market Statistics
    • What is my Property worth?
    • Mortgage Calculator
    • Vacation Rentals
    • Hurricane Preparedness
  • Top Neighborhoods
    • Collier County
    • Golf Communities
  • About
  • Contact
Close Bu

In [8]:
languages = []

for i in range(len(sampled_texts)):
    lang, conf = lang_id.identify_language(sampled_texts[i][1])
    languages.append((lang, round(conf, 4)))

print(languages)

INFO:cs336_data.lang_id.language_identifier:Identifying language. Doc preview: Barry Dunleavy logo

Default M
INFO:cs336_data.lang_id.language_identifier:fasttext output:
labels:('__label__en',)
confidence_arr:[0.84145159]
INFO:cs336_data.lang_id.language_identifier:Final label: en
Confidence:0.8414515852928162
INFO:cs336_data.lang_id.language_identifier:Identifying language. Doc preview: slotjp555.online - dewa6d ratu
INFO:cs336_data.lang_id.language_identifier:fasttext output:
labels:('__label__en',)
confidence_arr:[0.22240321]
INFO:cs336_data.lang_id.language_identifier:Final label: en
Confidence:0.22240321338176727
INFO:cs336_data.lang_id.language_identifier:Identifying language. Doc preview: Filter your search by Category
INFO:cs336_data.lang_id.language_identifier:fasttext output:
labels:('__label__ro',)
confidence_arr:[0.99011064]
INFO:cs336_data.lang_id.language_identifier:Final label: ro
Confidence:0.9901106357574463
INFO:cs336_data.lang_id.language_identifier:Identifying lang

[('en', 0.8415), ('en', 0.2224), ('ro', 0.9901), ('it', 0.9571), ('zh', 0.977), ('en', 0.931), ('en', 0.9547), ('ja', 0.7366), ('ja', 1.0), ('fr', 0.7938), ('ja', 0.9963), ('de', 0.236), ('fr', 0.9434), ('zh', 0.9969), ('zh', 0.9848), ('en', 0.1104), ('ro', 0.9667), ('pl', 0.9984), ('ja', 0.9999), ('zh', 0.954), ('en', 0.1104), ('en', 0.8583), ('es', 0.9252), ('en', 0.9478), ('en', 0.8123), ('en', 0.8776), ('en', 0.3653), ('de', 0.9954), ('en', 0.895), ('zh', 0.9309), ('de', 0.9929), ('zh', 0.9635), ('en', 0.1104), ('ja', 0.9957), ('zh', 0.9495), ('de', 0.8856), ('de', 0.9277), ('vi', 0.9933), ('en', 0.9198), ('id', 0.3941), ('ro', 0.8463), ('zh', 0.974), ('ja', 0.9888), ('en', 0.2823), ('en', 0.5092), ('en', 0.7236), ('ru', 0.9858), ('cs', 0.7851), ('ru', 0.9952), ('en', 0.6739)]


In [ ]:
lang_to_confidences = {}
for lang, conf in languages:
    print(lang)
    print(conf)
    if lang in lang_to_confidences:
        lang_to_confidences[lang].append(conf)
    else:
        lang_to_confidences[lang] = [conf] 

    print(lang_to_confidences)
    break

lang_to_confidences = {k:sorted(v, reverse=True) for k,v in lang_to_confidences.items()}

print(lang_to_confidences)

en
0.8415
{'en': [0.8415]}
{'e': ['n']}
